# Inference Configuration
Specify
- path

In [14]:
path = "../../output/protenn2/v5"


In [15]:
import json
import os.path
import pickle

from sklearn.preprocessing import LabelEncoder
from torch.utils.data import DataLoader

from src.protenn2.utils import get_train_val_test_paths, get_device

# get file paths

log_path = os.path.join(path, "log")

label_encoder_path = os.path.join(path, "label_encoder.pkl")
model_path = os.path.join(path, "best_model.pt")
with open(os.path.join(path, "params.json"), "r") as f:
    params = json.load(f)
if "input_folder" not in params:
    raise ValueError("input_folder must be specified")
dataset_path = os.path.join("../../", params["input_folder"])
train_path, val_path, test_path = get_train_val_test_paths(dataset_path)


In [16]:
from src.protenn2.utils import calculate_max_protein_length
from src.protenn2.dataset import CathPredPerResidueDataset, create_protein_collate_fn
from src.protenn2.analysis.cath_hierarchy_mapper import CATHHierarchyMapper

# Initialize objects

device = get_device()
with open(label_encoder_path, "rb") as f:
    label_encoder: LabelEncoder = pickle.load(f)
num_classes = len(label_encoder.classes_)
max_protein_length = calculate_max_protein_length(dataset_path)

test_dataset = CathPredPerResidueDataset(test_path, label_encoder=label_encoder,
                                         embedding_dir="../../data/embeddings/protein_embeddings_new")

collate_fn = create_protein_collate_fn(max_protein_length, test_dataset.padding_encoded_id)

test_dataloader = DataLoader(test_dataset, collate_fn=collate_fn)

mapper = CATHHierarchyMapper(label_encoder=label_encoder)

Using MPS (Apple Silicon GPU).
Max protein length: 599
Dataset initialized with 1317 unique proteins.


In [17]:
from src.protenn2.analysis.inference import run_inference_dummy, dummy_majority_classifier_per_protein

y_true_labels_list, y_pred_confidences_list = run_inference_dummy(dummy_classifier=dummy_majority_classifier_per_protein,
                                                                  dataloader=test_dataloader,
                                                                  padding_encoded_id=test_dataset.padding_encoded_id,
                                                                  majority_label=test_dataset.no_domain_encoded_id,
                                                                  num_classes=mapper.get_class_count(target_hierarchy="H"))

Running dummy inference on 1317 proteins...


Dummy Inference Progress: 100%|██████████| 1317/1317 [00:02<00:00, 605.22it/s]

Dummy inference complete. Processed 1317 proteins.


# Analysis Configuration

In [18]:
from src.protenn2.utils import call_domains_list

bootstrap_samples = 1000
post_process_kwargs = {"reporting_threshold": 0.2, "region_min_length": 20, "gaussian_sigma": 1}
post_process_func = call_domains_list
metrics_to_compute = ("accuracy", "f1_score", "jaccard_score", "recall_score", "precision_score",
                      "segment_overlap_score")

# metrics_to_compute = ("segment_overlap_score")

In [19]:
from src.protenn2.analysis.metrics import calculate_metrics_for_cath_levels

all_results = calculate_metrics_for_cath_levels(y_true_labels_list=y_true_labels_list,
                                                y_pred_confidences_list=y_pred_confidences_list, mapper=mapper,
                                                bootstrap_samples=bootstrap_samples,
                                                post_process_func=None,
                                                post_process_kwargs=None,
                                                metrics_to_compute=metrics_to_compute)

---- Computing Metrics for hierarchy: C
----- Accuracy Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:02<00:00, 443.61it/s]


{'mean': 0.3316539795007366, 'ci_lower': np.float64(0.3120246398071368), 'ci_upper': np.float64(0.3502452366131276), 'alpha': 0.05}
----- F1 Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:13<00:00, 74.89it/s]


{'mean': 0.16519961463249497, 'ci_lower': np.float64(0.1484108955119499), 'ci_upper': np.float64(0.1817028859085669), 'alpha': 0.05}
----- Jaccard Score Metrics -----


...

{'mean': 0.0, 'ci_lower': np.float64(0.0), 'ci_upper': np.float64(0.0), 'alpha': 0.05}
----- Precision Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:35<00:00, 27.93it/s]


{'mean': nan, 'ci_lower': np.float64(nan), 'ci_upper': np.float64(nan), 'alpha': 0.05}
----- Segment Overlap Score Metrics -----


Bootstrapping Progress: 100%|██████████| 1000/1000 [00:54<00:00, 18.37it/s]

{'mean': 0.0, 'ci_lower': np.float64(0.0), 'ci_upper': np.float64(0.0), 'alpha': 0.05}


# All results

In [20]:
all_results

{'raw_C': {'accuracy': {'mean': 0.3316539795007366,
   'ci_lower': np.float64(0.3120246398071368),
   'ci_upper': np.float64(0.3502452366131276),
   'alpha': 0.05},
  'f1_score': {'mean': 0.16519961463249497,
   'ci_lower': np.float64(0.1484108955119499),
   'ci_upper': np.float64(0.1817028859085669),
   'alpha': 0.05},
  'jaccard_score': {'mean': np.float64(0.0),
   'ci_lower': np.float64(0.0),
   'ci_upper': np.float64(0.0),
   'alpha': 0.05},
  'recall_score': {'mean': 0.0,
   'ci_lower': np.float64(0.0),
   'ci_upper': np.float64(0.0),
   'alpha': 0.05},
  'precision_score': {'mean': nan,
   'ci_lower': np.float64(nan),
   'ci_upper': np.float64(nan),
   'alpha': 0.05},
  'segment_overlap_score': {'mean': 0.0,
   'ci_lower': np.float64(0.0),
   'ci_upper': np.float64(0.0),
   'alpha': 0.05}},
 'raw_A': {'accuracy': {'mean': 0.3316539795007366,
   'ci_lower': np.float64(0.3120246398071368),
   'ci_upper': np.float64(0.3502452366131276),
   'alpha': 0.05},
  'f1_score': {'mean': 0.16

In [21]:
with open(os.path.join(path, "baseline_test_metrics.json"), "w") as f:
    json.dump(all_results, f)

In [ ]:
from src.protenn2.analysis.plot import plot_metric_with_ci

plt = plot_metric_with_ci(all_results, "accuracy")
plt.show()
plt = plot_metric_with_ci(all_results, "jaccard_score")
plt.show()
plt = plot_metric_with_ci(all_results, "f1_score")
plt.show()

In [ ]:
plt = plot_metric_with_ci(all_results, "segment_overlap_score")
plt.show()